# MFT Theory - Template Notebook

One-line setup, config generation, truncate cell for cluster export.

In [ ]:
import sys, os
import pickle
import numpy as np
sys.path.append(os.path.join(os.path.expanduser('~'), 'mft-theory'))

from scripts.notebook_setup import setup_mft_theory
setup = setup_mft_theory(use_gpu=False, import_cluster=True, import_empirics=True)
device = setup['device']
device_idx = setup['device_idx']
to_torch = setup['to_torch']

In [ ]:
### --- SET UP ALL CONFIGS --- ###
from itertools import product
n_seeds = 1
macro_configs = config_generator()

micro_configs = tuple(product(macro_configs, list(range(n_seeds))))
prototype = False

### --- SELECT PARTICULAR CONFIG --- ###
try:
    i_job = int(os.environ['SLURM_ARRAY_TASK_ID']) - 1
except KeyError:
    i_job = 0
    prototype = True
params, i_seed = micro_configs[i_job]
i_config = i_job//n_seeds

new_random_seed_per_condition = True
if new_random_seed_per_condition:
    np.random.seed(i_job)
else: #Match random seeds across conditions
    np.random.seed(i_seed)

In [ ]:
processed_data = np.array([])

In [ ]:
### --- SAVE RESULTS -- ###
result = {'sim': None, 'dim_emp': None,
          'i_seed': i_seed, 'config': params,
          'i_config': i_config, 'i_job': i_job}
try:
    result['processed_data'] = processed_data
except NameError:
    pass
    
try:
    save_dir = os.environ['SAVEDIR']
    if not os.path.exists(save_dir):
        os.mkdir(save_dir)
    save_path = os.path.join(save_dir, 'result_{}'.format(i_job))

    with open(save_path, 'wb') as f:
        pickle.dump(result, f)
        f.flush()
        os.fsync(f.fileno())
except KeyError:
    pass
except Exception as e:
    print(f"Error saving result: {e}")
    raise

In [ ]:
###Truncate file above
file_name = 'my_analysis'
job_name = 'my_job'
project_dir = '/home/om2382/low-rank-dims/'
main_script_path = os.path.join(project_dir, 'cluster_main_scripts', job_name + '.py')
get_ipython().run_cell_magic('javascript', '', 'IPython.notebook.save_notebook()')
get_ipython().system('jupyter nbconvert --to script --no-prompt {}.ipynb'.format(file_name))
get_ipython().system('awk "/###Truncate/ {{exit}} {{print}}" {}.py'.format(file_name))
get_ipython().system('sed -i "/###Truncate/Q" {}.py'.format(file_name))
get_ipython().system('mv {}.py {}'.format(file_name, main_script_path))

In [ ]:
write_job_file(job_name, py_file_name='{}.py'.format(job_name), mem=64, n_hours=24, n_gpus=1,
               results_subdir='misc')
job_script_path = os.path.join(project_dir, 'job_scripts', job_name + '.s')
n_jobs = len(micro_configs)
submit_job(job_script_path, n_jobs, execute=False,
           results_subdir='misc', lkumar=False)

In [ ]:
results = unpack_processed_data(job_script_path, results_subdir='misc')